In [ ]:
# Install required libraries
!pip -q install transformers accelerate bitsandbytes qwen-vl-utils scikit-learn pandas numpy pillow tqdm scipy joblib

In [ ]:
import os
import gc
import re
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from scipy.sparse import hstack, csr_matrix, save_npz, load_npz
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

1. Set dataset paths
Change these paths according to your Google Drive folder structure.

Expected CSV columns:

image_id
context
category
Expected image folders:

Train images
Validation images
Test images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/"

TRAIN_CSV = os.path.join(DATASET_ROOT, "Disaster_train.csv")
VAL_CSV   = os.path.join(DATASET_ROOT, "Disaster_validation.csv")
TEST_CSV  = os.path.join(DATASET_ROOT, "Disaster_test.csv")

TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, "Train")
VAL_IMG_DIR   = os.path.join(DATASET_ROOT, "Validation")
TEST_IMG_DIR  = os.path.join(DATASET_ROOT, "Test")

RESULT_DIR = os.path.join(DATASET_ROOT, "feature_classifier_results")
os.makedirs(RESULT_DIR, exist_ok=True)

print("Result folder:", RESULT_DIR)

2. Fix Bangla text and normalize labels
Sample text is in mojibake:

à¦à¦¡à¦¼à§‡...

That usually means Bangla text was decoded incorrectly. This notebook tries to repair it before training.

In [ ]:
CLASSES = [
    "Earthquake",
    "Flood",
    "Landslides",
    "Wildfire",
    "Tropical Storm",
    "Drought",
    "Human Damage",
    "Non Disaster"
]

LABEL_FIX = {
    "Earthquake": "Earthquake",
    "Earthquakes": "Earthquake",

    "Flood": "Flood",
    "Floods": "Flood",

    "Landslide": "Landslides",
    "Landslides": "Landslides",

    "Wildfire": "Wildfire",
    "Wildfires": "Wildfire",

    "Tropical Storm": "Tropical Storm",
    "Tropical Storms": "Tropical Storm",

    "Drought": "Drought",
    "Droughts": "Drought",

    "Human Damage": "Human Damage",
    "Human-Damage": "Human Damage",

    "Non Disaster": "Non Disaster",
    "Non-Disaster": "Non Disaster",
    "Non-disaster": "Non Disaster",
}

def fix_mojibake_text(x):
    if pd.isna(x):
        return ""

    x = str(x)

    if "à¦" in x or "à§" in x:
        try:
            return x.encode("latin1").decode("utf-8")
        except Exception:
            return x

    return x

def clean_text(x):
    x = fix_mojibake_text(x)
    x = x.replace("\n", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def get_image_path(image_id, image_dir):
    image_id = str(image_id).strip()
    base = os.path.join(image_dir, image_id)

    for ext in [".jpg", ".jpeg", ".png", ".webp", ".JPG", ".PNG", ""]:
        p = base + ext
        if os.path.exists(p):
            return p

    return base

def load_split(csv_path, image_dir):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found at: {csv_path}. Please ensure the dataset is correctly unzipped and the path is accurate.")

    df = pd.read_csv(csv_path)

    required_cols = ["image_id", "context", "category"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in {csv_path}: {missing_cols}")

    df["context"] = df["context"].apply(clean_text)
    df["category"] = df["category"].astype(str).str.strip().map(LABEL_FIX)

    if df["category"].isna().sum() > 0:
        print("Unmapped labels:")
        print(df[df["category"].isna()].head(10))
        raise ValueError("Some labels could not be mapped. Update LABEL_FIX.")

    df["image_path"] = df["image_id"].apply(lambda x: get_image_path(x, image_dir))

    missing_images = (~df["image_path"].apply(os.path.exists)).sum()

    print("\n", os.path.basename(csv_path))
    print("Rows:", len(df))
    print("Missing images:", missing_images)
    print(df["category"].value_counts())

    if missing_images > 0:
        print("\nExamples of missing images:")
        print(df[~df["image_path"].apply(os.path.exists)][["image_id", "image_path"]].head())

    return df

# Correcting DATASET_ROOT based on previous file listing. The actual dataset files (CSVs and image folders)
# are likely located inside 'AI_Project/BanglaCalamityMMD'.
DATASET_ROOT = "/content/drive/MyDrive/BanglaCalamityMMD"

TRAIN_CSV = os.path.join(DATASET_ROOT, "Disaster_train.csv")
VAL_CSV   = os.path.join(DATASET_ROOT, "Disaster_validation.csv")
TEST_CSV  = os.path.join(DATASET_ROOT, "Disaster_test.csv")

TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, "Train")
VAL_IMG_DIR   = os.path.join(DATASET_ROOT, "Validation")
TEST_IMG_DIR  = os.path.join(DATASET_ROOT, "Test")

train_df = load_split(TRAIN_CSV, TRAIN_IMG_DIR)
val_df   = load_split(VAL_CSV, VAL_IMG_DIR)
test_df  = load_split(TEST_CSV, TEST_IMG_DIR)

print("\nSample cleaned text:")
print(train_df[["image_id", "context", "category"]].head(3))

In [ ]:
def evaluate(y_true, y_pred, name):
    print("\n" + "="*80)
    print(name)
    print("="*80)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"Accuracy        : {acc*100:.2f}%")
    print(f"Macro Precision : {prec*100:.2f}%")
    print(f"Macro Recall    : {rec*100:.2f}%")
    print(f"Macro F1        : {f1*100:.2f}%")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=CLASSES, digits=4, zero_division=0))

    print("\nConfusion Matrix:")
    print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=CLASSES), index=CLASSES, columns=CLASSES))

    return {
        "accuracy": acc,
        "macro_precision": prec,
        "macro_recall": rec,
        "macro_f1": f1
    }

## Part C: Gemma Text Feature Adapter + Bangla Text Classifier

This section loads a Gemma model in 4-bit and extracts text embeddings. It's used as a text feature extractor, similar to how TF-IDF is used for text only in the BLIP2 section.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import huggingface_hub

huggingface_hub.login()

torch.cuda.empty_cache()
gc.collect()

gemma_quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

GEMMA_ID = "google/gemma-3-4b-it"

gemma_tokenizer = AutoTokenizer.from_pretrained(GEMMA_ID, trust_remote_code=True)

gemma_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_ID,
    quantization_config=gemma_quant,
    device_map="auto",
    trust_remote_code=True
)

gemma_model.eval()

print("Gemma loaded.")

In [ ]:
import contextlib

def gemma_get_embedding(text, max_text_chars=1000):
    text = clean_text(text)[:max_text_chars]
    inputs = gemma_tokenizer(text, return_tensors="pt", truncation=True, padding=True).to("cuda")

    # ── Fix 1: mean pooling instead of last token ──
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
        outputs = gemma_model(**inputs, output_hidden_states=True, return_dict=True)
        emb = outputs.hidden_states[-1].float().mean(dim=1).cpu().numpy()[0]  # mean pool

    del inputs, outputs
    torch.cuda.empty_cache()
    return emb

def build_gemma_embeddings(df, save_path):
    if os.path.exists(save_path):
        print("Loading saved embeddings:", save_path)
        return np.load(save_path)

    embeddings = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Gemma embeddings"):
        emb = gemma_get_embedding(row["context"])
        embeddings.append(emb)

    embeddings = np.vstack(embeddings).astype("float32")
    np.save(save_path, embeddings)
    print("Saved:", save_path)
    return embeddings

In [ ]:
import os
import numpy as np

gemma_train_path = os.path.join(RESULT_DIR, "gemma_train_embeddings.npy")
gemma_val_path   = os.path.join(RESULT_DIR, "gemma_val_embeddings.npy")
gemma_test_path  = os.path.join(RESULT_DIR, "gemma_test_embeddings.npy")

Xg_train = build_gemma_embeddings(train_df, gemma_train_path)
Xg_val   = build_gemma_embeddings(val_df, gemma_val_path)
Xg_test  = build_gemma_embeddings(test_df, gemma_test_path)

print("Gemma embedding shapes:")
print(Xg_train.shape, Xg_val.shape, Xg_test.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# Bangla text features for Gemma section
gemma_text_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 6),
    max_features=100000,
    min_df=2,
    sublinear_tf=True
)

Xt_train_g = gemma_text_vectorizer.fit_transform(train_df["context"])
Xt_val_g   = gemma_text_vectorizer.transform(val_df["context"])
Xt_test_g  = gemma_text_vectorizer.transform(test_df["context"])

# Combine Gemma features + Bangla text features
X_train_g = hstack([csr_matrix(Xg_train), Xt_train_g])
X_val_g   = hstack([csr_matrix(Xg_val), Xt_val_g])
X_test_g  = hstack([csr_matrix(Xg_test), Xt_test_g])

# Reuse y_train, y_val, y_test from previous sections as they are based on dataframes
# y_train = train_df["category"].values
# y_val   = val_df["category"].values
# y_test  = test_df["category"].values

print("Final Gemma feature shapes:")
print(X_train_g.shape, X_val_g.shape, X_test_g.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

best_gemma_model = None
best_gemma_f1 = -1
best_gemma_c = None

# Prepare labels
y_train = train_df["category"].values
y_val   = val_df["category"].values
y_test  = test_df["category"].values

# Scaling combined features ---
scaler_g = StandardScaler(with_mean=False)
X_train_g_scaled = scaler_g.fit_transform(X_train_g)
X_val_g_scaled   = scaler_g.transform(X_val_g)
X_test_g_scaled  = scaler_g.transform(X_test_g)

print("Starting Hyperparameter Tuning with Expanded Search...")

# We use much smaller C values because the high feature count (77k+)
# makes the model very prone to 'over-fitting' the separating plane.
for c in [0.00001, 0.0001, 0.0005, 0.001, 0.01, 0.1]:
    clf = LinearSVC(
        C=c,
        class_weight="balanced",
        max_iter=20000,
        dual="auto", # 'auto' or False is better when n_features > n_samples
        tol=1e-4,
        random_state=SEED
    )

    clf.fit(X_train_g_scaled, y_train)
    val_pred = clf.predict(X_val_g_scaled)

    val_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
    val_acc = accuracy_score(y_val, val_pred)

    print(f"C={c:<8} | Val Accuracy={val_acc*100:.2f}% | Val Macro F1={val_f1*100:.2f}%")

    if val_f1 > best_gemma_f1:
        best_gemma_f1 = val_f1
        best_gemma_c = c
        best_gemma_model = clf

print(f"\nBest Gemma C: {best_gemma_c}")

# Immediate Test Check
test_pred = best_gemma_model.predict(X_test_g_scaled)
test_acc = accuracy_score(y_test, test_pred)
print(f"Final Enhanced Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
# Predict on scaled test set
gemma_test_pred = best_gemma_model.predict(X_test_g_scaled)

gemma_metrics = evaluate(
    y_true=y_test,
    y_pred=gemma_test_pred,
    name="Gemma Text Feature Extractor + Bangla Text Classifier"
)

gemma_out = test_df[["image_id", "context", "category", "image_path"]].copy()
gemma_out["prediction"] = gemma_test_pred
gemma_out.to_csv(os.path.join(RESULT_DIR, "gemma_feature_classifier_predictions.csv"), index=False)

# Save model, scaler, and the correct text vectorizer
joblib.dump(best_gemma_model,  os.path.join(RESULT_DIR, "gemma_linearsvc.joblib"))
joblib.dump(scaler_g,          os.path.join(RESULT_DIR, "gemma_scaler.joblib"))
joblib.dump(gemma_text_vectorizer, os.path.join(RESULT_DIR, "gemma_text_vectorizer.joblib"))

print("Saved Gemma predictions and classifier.")

### Clear Gemma from GPU memory before loading next model

Run this before the next section to free up resources.

In [ ]:
del gemma_model
del gemma_tokenizer
torch.cuda.empty_cache()
gc.collect()

print("Gemma removed from GPU memory.")

## LLaVA Feature Adapter + Bangla Text Classifier

This section loads a LLaVA model in 4-bit and extracts hidden-state embeddings, similar to Qwen2-VL. It combines both visual and textual features.

In [ ]:
!pip install -U bitsandbytes>=0.46.1

from transformers import LlavaForConditionalGeneration, AutoTokenizer, AutoProcessor, BitsAndBytesConfig
import torch
import gc

# Ensure previous models are cleared from memory if they exist, to prevent OOM errors.
try:
    if 'gemma_model' in locals() and gemma_model is not None:
        del gemma_model
except NameError:
    pass
try:
    if 'gemma_tokenizer' in locals() and gemma_tokenizer is not None:
        del gemma_tokenizer
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

# Configure 4-bit quantization for efficient memory usage and faster computation
llava_quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for computation with 4-bit quantization
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4" # NormalFloat4 quantization type
)

LLAVA_ID = "llava-hf/llava-1.5-7b-hf" # Model identifier for LLaVA

# Load the processor (tokenizer and image processor) and the LLaVA model
llava_processor = AutoProcessor.from_pretrained(LLAVA_ID, trust_remote_code=True)
llava_model = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_ID,
    quantization_config=llava_quant,
    device_map="auto", # Automatically map the model to available devices (e.g., GPU)
    trust_remote_code=True
)

llava_model.eval() # Set the model to evaluation mode

print("LLaVA loaded.")

In [ ]:
def llava_get_embeddings_batch(df, batch_size=16, save_path="embeddings.npy"):
    """
    Extract LLaVA embeddings from images + text context in batches.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    if os.path.exists(save_path):
        print(f"Loading saved embeddings: {save_path}")
        return np.load(save_path)

    embeddings = []
    failed_indices = []

    for batch_idx in tqdm(
        range(0, len(df), batch_size),
        total=(len(df) // batch_size) + (1 if len(df) % batch_size else 0),
        desc="LLaVA batch embeddings"
    ):
        batch = df.iloc[batch_idx : batch_idx + batch_size]
        images = []
        texts = []

        for local_idx, (_, row) in enumerate(batch.iterrows()):
            global_idx = batch_idx + local_idx
            if os.path.exists(row["image_path"]):
                try:
                    img = Image.open(row["image_path"]).convert("RGB")
                    img.thumbnail((224, 224))
                    images.append(img)
                    texts.append(clean_text(row["context"])[:250])
                except:
                    failed_indices.append(global_idx)
            else:
                failed_indices.append(global_idx)

        if not images:
            continue

        prompts = [
            f"USER: <image>\nClassify the disaster type. Context: {text}\nASSISTANT:"
            for text in texts
        ]

        try:
            inputs = llava_processor(
                text=prompts,
                images=images,
                return_tensors="pt",
                padding=True
            ).to(device)

            with torch.no_grad():
                outputs = llava_model(
                    **inputs,
                    output_hidden_states=True,
                    return_dict=True
                )
                # Extract embedding from last hidden state
                batch_embs = outputs.hidden_states[-1][:, -1, :].detach().float().cpu().numpy()
                embeddings.append(batch_embs)

            del inputs, outputs
            if device == "cuda":
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"Batch failed: {e}")
            continue

    if not embeddings:
        raise ValueError("No embeddings extracted.")

    embeddings = np.vstack(embeddings).astype("float32")
    np.save(save_path, embeddings)
    return embeddings

In [ ]:
import os
import numpy as np

llava_train_path = os.path.join(RESULT_DIR, "llava_train_embeddings.npy")
llava_val_path   = os.path.join(RESULT_DIR, "llava_val_embeddings.npy")
llava_test_path  = os.path.join(RESULT_DIR, "llava_test_embeddings.npy")

# Re-running with the fixed function
Xl_train = llava_get_embeddings_batch(train_df, batch_size=8, save_path=llava_train_path)
Xl_val   = llava_get_embeddings_batch(val_df, batch_size=8, save_path=llava_val_path)
Xl_test  = llava_get_embeddings_batch(test_df, batch_size=8, save_path=llava_test_path)

print("LLaVA embedding shapes:")
print(Xl_train.shape, Xl_val.shape, Xl_test.shape)

In [ ]:
import os
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# Check if LLaVA embeddings are already loaded, if not, load them from disk
# This makes the cell robust to restarts or out-of-order execution where Xl_train etc. might be missing.
if 'Xl_train' not in locals() and 'Xl_train' not in globals():
    print("LLaVA embeddings (Xl_train, Xl_val, Xl_test) not found in memory. Attempting to load from disk.")
    llava_train_path = os.path.join(RESULT_DIR, "llava_train_embeddings.npy")
    llava_val_path   = os.path.join(RESULT_DIR, "llava_val_embeddings.npy")
    llava_test_path  = os.path.join(RESULT_DIR, "llava_test_embeddings.npy")

    try:
        Xl_train = np.load(llava_train_path)
        Xl_val   = np.load(llava_val_path)
        Xl_test  = np.load(llava_test_path)
        print("Successfully loaded LLaVA embeddings from disk.")
    except FileNotFoundError:
        print("LLaVA embedding files not found. Please ensure cell '34827582' (build_llava_embeddings) has been executed successfully to generate and save these files.")
        raise # Re-raise to halt execution if files are truly missing
    except Exception as e:
        print(f"An unexpected error occurred while loading LLaVA embeddings: {e}")
        raise

# Initialize TF-IDF Vectorizer for Bangla text features
# 'char_wb' analyzer captures character n-grams within word boundaries, useful for morphologically rich languages like Bangla.
llava_text_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5), # Consider character n-grams from 2 to 5
    max_features=60000, # Limit to the top 60,000 features
    min_df=2, # Ignore terms that appear in less than 2 documents
    sublinear_tf=True # Apply sublinear TF scaling (1 + log(tf)) to downweight frequent terms
)

# Fit the vectorizer on training data and transform all splits
Xt_train_l = llava_text_vectorizer.fit_transform(train_df["context"])
Xt_val_l   = llava_text_vectorizer.transform(val_df["context"])
Xt_test_l  = llava_text_vectorizer.transform(test_df["context"])

# Combine LLaVA features (image+text embeddings) with TF-IDF Bangla text features
X_train_l = hstack([csr_matrix(Xl_train), Xt_train_l])
X_val_l   = hstack([csr_matrix(Xl_val), Xt_val_l])
X_test_l  = hstack([csr_matrix(Xl_test), Xt_test_l])

# Reuse y_train, y_val, y_test from previous sections as they are based on dataframes
# y_train = train_df["category"].values
# y_val   = val_df["category"].values
# y_test  = test_df["category"].values

print("Final LLaVA feature shapes:")
print(X_train_l.shape, X_val_l.shape, X_test_l.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

best_llava_model = None
best_llava_f1 = -1
best_llava_c = None

# Initialize target labels from dataframes
y_train = train_df["category"].values
y_val   = val_df["category"].values
y_test  = test_df["category"].values

# Scale features to help convergence (with_mean=False for sparse matrices)
scaler_l = StandardScaler(with_mean=False)
X_train_l_scaled = scaler_l.fit_transform(X_train_l)
X_val_l_scaled   = scaler_l.transform(X_val_l)

for c in [0.001, 0.005, 0.01, 0.025, 0.05, 0.1, 0.5, 1]:
    clf = LinearSVC(
        C=c,
        class_weight="balanced",
        max_iter=20000, # Increased iterations
        random_state=SEED
    )

    # Train and evaluate
    clf.fit(X_train_l_scaled, y_train)
    val_pred = clf.predict(X_val_l_scaled)

    val_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
    val_acc = accuracy_score(y_val, val_pred)

    print(f"C={c:<6} | Val Accuracy={val_acc*100:.2f}% | Val Macro F1={val_f1*100:.2f}%")

    if val_f1 > best_llava_f1:
        best_llava_f1 = val_f1
        best_llava_c = c
        best_llava_model = clf

print("\nBest LLaVA C:", best_llava_c)

In [ ]:
# Use the best trained LLaVA model to predict on the test set
llava_test_pred = best_llava_model.predict(X_test_l)

# Evaluate the model's performance on the test set
llava_metrics = evaluate(
    y_true=y_test,
    y_pred=llava_test_pred,
    name="LLaVA Feature Extractor + Bangla Text Classifier" # Name for the evaluation report
)

# Save predictions to a CSV file for further analysis
llava_out = test_df[["image_id", "context", "category", "image_path"]].copy()
llava_out["prediction"] = llava_test_pred
llava_out.to_csv(os.path.join(RESULT_DIR, "llava_feature_classifier_predictions.csv"), index=False)

# Save the best classifier model and the TF-IDF vectorizer
# This allows for later reuse without retraining
joblib.dump(best_llava_model, os.path.join(RESULT_DIR, "llava_linearsvc.joblib"))
joblib.dump(llava_text_vectorizer, os.path.join(RESULT_DIR, "llava_text_vectorizer.joblib"))

print("Saved LLaVA predictions and classifier.")

### Clear LLaVA from GPU memory before loading next model (if any)

Run this to free up resources.


In [ ]:
del llava_model
del llava_processor
torch.cuda.empty_cache()
gc.collect()

print("LLaVA removed from GPU memory.")


In [ ]:
### Summary for Qwen and BLIP2 (SKIPPED)

In [ ]:
summary_rows = []
try:
    # Append Gemma metrics to the summary table if available
    summary_rows.append({
        "model": "Gemma features + Bangla TF-IDF + LinearSVC",
        **gemma_metrics
    })
except NameError:
    pass

try:
    # Append LLaVA metrics to the summary table if available
    summary_rows.append({
        "model": "LLaVA features + Bangla TF-IDF + LinearSVC",
        **llava_metrics
    })
except NameError:
    pass

# Display the summary table if any metrics were collected
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
    # Save the summary to a CSV file
    summary_df.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"), index=False)
else:
    print("No metrics found yet. Run LLaVA section first.")